# Northwind Sales Analysis

Análise de dados da empresa fictícia Northwind Traders utilizando MySQL, pandas e seaborn.

**Análises realizadas:**
- Pedidos por cliente
- Top 10 produtos mais vendidos
- Receita total por mês
- Margem de lucro por produto
- Funcionário que gerou mais receita

## 1. Importações e Conexão com o Banco

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Conectando ao banco de dados
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="SUA_SENHA_AQUI",
    database="northwind"
)

print("Conexão bem sucedida!" if conn.is_connected() else "Falhou!")

## 2. Pedidos por Cliente

Identifica os clientes mais ativos e revela clientes cadastrados que nunca realizaram uma compra.

In [ ]:
query_clientes = "SELECT c.first_name, c.last_name, COUNT(o.id) AS orders_quantity FROM customers c LEFT JOIN orders o ON c.id = o.customer_id GROUP BY c.id, c.first_name, c.last_name ORDER BY orders_quantity DESC"

df_clientes = pd.read_sql(query_clientes, conn)
print(df_clientes)

## 3. Top 10 Produtos Mais Vendidos

Ranking dos produtos com maior volume de unidades vendidas.

In [ ]:
query_produtos = "SELECT p.product_name, SUM(d.quantity) AS total_quantity FROM products p INNER JOIN order_details d ON p.id = d.product_id GROUP BY p.product_name ORDER BY total_quantity DESC LIMIT 10"

df_produtos = pd.read_sql(query_produtos, conn)
print(df_produtos)

In [ ]:
ax = sns.barplot(x='total_quantity', y='product_name', data=df_produtos)
ax.figure.set_size_inches(12, 6)
ax.set_title('Top 10 Produtos Mais Vendidos')
ax.set_xlabel('Quantidade Vendida')
ax.set_ylabel('Produto')
sns.despine()
plt.tight_layout()
plt.savefig('charts/top_produtos.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Receita Total por Mês

Evolução da receita ao longo do tempo. Permite identificar sazonalidade e picos de vendas.

In [ ]:
query_receita = "SELECT DATE_FORMAT(o.order_date, '%Y-%m'), SUM(d.quantity * d.unit_price) AS total_revenue FROM orders o LEFT JOIN order_details d ON o.id = d.order_id GROUP BY DATE_FORMAT(o.order_date, '%Y-%m') ORDER BY DATE_FORMAT(o.order_date, '%Y-%m')"

df_receita = pd.read_sql(query_receita, conn)
df_receita = df_receita.rename(columns={"DATE_FORMAT(o.order_date, '%Y-%m')": 'year_month'})
print(df_receita)

In [ ]:
ax = sns.barplot(x='year_month', y='total_revenue', data=df_receita)
ax.figure.set_size_inches(12, 6)
ax.set_title('Receita Mensal')
ax.set_xlabel('Mes/Ano')
ax.set_ylabel('Receita Total')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
sns.despine()
plt.tight_layout()
plt.savefig('charts/receita_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Margem de Lucro por Produto

Calcula a margem percentual de cada produto. Fórmula: (preco_venda - custo) / preco_venda * 100

In [ ]:
query_margem = "SELECT product_name, standard_cost, list_price, ((list_price - standard_cost) / list_price) * 100 AS margin_pct FROM products ORDER BY margin_pct DESC"

df_margem = pd.read_sql(query_margem, conn)
print(df_margem)

In [ ]:
ax = sns.barplot(x='margin_pct', y='product_name', data=df_margem.head(10))
ax.figure.set_size_inches(12, 6)
ax.set_title('Top 10 Produtos com Maior Margem de Lucro')
ax.set_xlabel('Margem de Lucro (%)')
ax.set_ylabel('Produto')
sns.despine()
plt.tight_layout()
plt.savefig('charts/margem_lucro.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Funcionário que Gerou Mais Receita

Avalia a performance individual da equipe de vendas com base na receita gerada.

In [ ]:
query_funcionarios = "SELECT e.first_name, e.last_name, SUM(d.quantity * d.unit_price) AS total_revenue FROM employees e LEFT JOIN orders o ON o.employee_id = e.id JOIN order_details d ON d.order_id = o.id GROUP BY e.first_name, e.last_name ORDER BY total_revenue DESC"

df_funcionarios = pd.read_sql(query_funcionarios, conn)
print(df_funcionarios)

## 7. Exportar Relatório Excel

In [ ]:
with pd.ExcelWriter('northwind_relatorio.xlsx', engine='openpyxl') as writer:
    df_clientes.to_excel(writer, sheet_name='Pedidos por Cliente', index=False)
    df_produtos.to_excel(writer, sheet_name='Top Produtos', index=False)
    df_receita.to_excel(writer, sheet_name='Receita Mensal', index=False)
    df_margem.to_excel(writer, sheet_name='Margem de Lucro', index=False)
    df_funcionarios.to_excel(writer, sheet_name='Performance Funcionarios', index=False)

print("Relatorio salvo com sucesso!")